In [ ]:
pip install pyspark

In [39]:
import pyspark
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.storagelevel import StorageLevel

In [ ]:
spark = SparkSession.builder.getOrCreate()

In [ ]:
df_video = spark.read.parquet('videos-preparados.snappy.parquet', header=True, inferSchema=True)
df_comments = spark.read.parquet('videos-comments-tratados.snappy.parquet', header=True, inferSchema=True)

In [29]:
df_video.createOrReplaceTempView('video_temp')
df_comments.createOrReplaceTempView('comments_temp')
#Cria tabelas temporárias

In [ ]:
df_video.show(5)
df_comments.show(5)

+--------------------+-----------+------------+-------+------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|               Title|   Video ID|Published At|Keyword| Likes|Comments|   Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|
+--------------------+-----------+------------+-------+------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|ASMR MUKBANG DOUB...|--ZI0dSbbNU|  2020-04-18|mukbang|378858|   18860|17975269|   18372987|2020|    4|         30.0|[0.6985786560867407]|[0.02303716158264...|[378858.0,1.79752...|
|Deadly car bomb d...|--hxd1CrOqg|  2022-08-22|   news|  6379|    4853|  808787|     820019|2022|    8|         37.0|[0.8936407990235931]|[3.87946679100418...|[6379.0,808787.0,...|
|How Biden&#39;s s...|--ixiTypG8g|  2022-08-24|   news|  1029|    2347|   97434|     100810|202

In [ ]:
join_video_comments = spark.sql("""
    Select *
    From video_temp v
    Join comments_temp c on v.`Video ID` = c.`Video ID`
""")
join_video_comments.show(5)
#Uni as tabelas

+--------------------+-----------+------------+-------+-----+--------+------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+-------+-----+--------+------+-----------+----+--------------------+---------+-------------+
|               Title|   Video ID|Published At|Keyword|Likes|Comments| Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|   Video ID|               Title|Published At|Keyword|Likes|Comments| Views|Interaction|Year|             Comment|Sentiment|Likes Comment|
+--------------------+-----------+------------+-------+-----+--------+------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+-------+-----+--------+------+-----------+----+--------------------+---------+-------------+
|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23|   te

In [ ]:
print('Número de partições padrão df_video:', df_video.rdd.getNumPartitions())

df_video = spark.read.parquet('videos-preparados.snappy.parquet', header=True, inferSchema=True).repartition(5)
print('Número de partições com repartition df_video:', df_video.rdd.getNumPartitions())

Número de partições padrão df_video: 1
Número de partições com repartition df_video: 5


In [ ]:
print('Número de partições padrão df_comments:', df_comments.rdd.getNumPartitions())

df_comments = spark.read.parquet('videos-comments-tratados.snappy.parquet', header=True, inferSchema=True).repartition(5)
print('Número de partições com repartition df_comments:', df_comments.rdd.getNumPartitions())

Número de partições padrão df_comments: 1
Número de partições com repartition df_comments: 5


In [ ]:
df_video_2p = df_video.coalesce(2)
print('Número de partições com coalesce df_video:', df_video_2p.rdd.getNumPartitions())

Número de partições com coalesce df_video: 2


In [ ]:
df_comments_2p = df_comments.coalesce(2)
print('Número de partições com coalesce df_comments:', df_comments_2p.rdd.getNumPartitions())

Número de partições com coalesce df_comments: 2


In [31]:
df_video_2p.createOrReplaceTempView('video_temp_2')
df_comments_2p.createOrReplaceTempView('comments_temp_2')

In [32]:
join_video_comments_2p = spark.sql("""
    Select *
    From video_temp_2 v
    Join comments_temp_2 c on v.`Video ID` = c.`Video ID`
""")
join_video_comments_2p.show(5)

+--------------------+-----------+------------+----------------+-----+--------+------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+----------------+-----+--------+------+-----------+----+--------------------+---------+-------------+
|               Title|   Video ID|Published At|         Keyword|Likes|Comments| Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|   Video ID|               Title|Published At|         Keyword|Likes|Comments| Views|Interaction|Year|             Comment|Sentiment|Likes Comment|
+--------------------+-----------+------------+----------------+-----+--------+------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+----------------+-----+--------+------+-----------+----+--------------------+---------+-------------

In [40]:
start_time = time.time()
join_video_comments = spark.sql("""
    Select *
    From video_temp v
    Join comments_temp c on v.`Video ID` = c.`Video ID`
""")
join_video_comments.show(5)

end_time = time.time()
print(end_time - start_time)

print('Plano de Execução Join:')
join_video_comments.explain()

#Verifica o tempo de execução e explica o caminho percorrido para leitura do DataFrame com as partições original

+--------------------+-----------+------------+----------------+-----+--------+------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+----------------+-----+--------+------+-----------+----+--------------------+---------+-------------+
|               Title|   Video ID|Published At|         Keyword|Likes|Comments| Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|   Video ID|               Title|Published At|         Keyword|Likes|Comments| Views|Interaction|Year|             Comment|Sentiment|Likes Comment|
+--------------------+-----------+------------+----------------+-----+--------+------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+----------------+-----+--------+------+-----------+----+--------------------+---------+-------------

In [41]:
start_time = time.time()
join_video_comments_2p = spark.sql("""
    Select *
    From video_temp_2 v
    Join comments_temp_2 c on v.`Video ID` = c.`Video ID`
""")
join_video_comments_2p.show(5)

end_time = time.time()
print(end_time - start_time)

print('Plano de Execução Join 2 Partições:')
join_video_comments_2p.explain()

##Verifica o tempo de execução e explica o caminho percorrido para leitura do DataFrame dividido em 2 partições, se mostrando mais eficiente.

+--------------------+-----------+------------+----------------+-----+--------+------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+----------------+-----+--------+------+-----------+----+--------------------+---------+-------------+
|               Title|   Video ID|Published At|         Keyword|Likes|Comments| Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|   Video ID|               Title|Published At|         Keyword|Likes|Comments| Views|Interaction|Year|             Comment|Sentiment|Likes Comment|
+--------------------+-----------+------------+----------------+-----+--------+------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+----------------+-----+--------+------+-----------+----+--------------------+---------+-------------

In [43]:
start_time = time.time()
join_video_comments_otimizado = spark.sql("""
    SELECT v.`Video ID`, v.Title, v.Views, c.Comments, c.Likes, c.Interaction
    From video_temp_2 v
    Join comments_temp_2 c on v.`Video ID` = c.`Video ID`
""")
join_video_comments_otimizado.show(10)

end_time = time.time()
print(end_time - start_time)
print('Plano de Execução Join otimizado:')
join_video_comments_otimizado.explain()

#Mostra o tempo de execução e explica o caminho feito para unir as duas tabelas com os dados mais importantes

+-----------+--------------------+--------+--------+------+-----------+
|   Video ID|               Title|   Views|Comments| Likes|Interaction|
+-----------+--------------------+--------+--------+------+-----------+
|szlhtejAMyM|Craziest &quot;Th...|15110047|    3715|132236|   15245998|
|szlhtejAMyM|Craziest &quot;Th...|15110047|    3715|132236|   15245998|
|y3KQzBnB7-U|Vocal Coaches Rea...|   38560|     225|  3645|      42430|
|y3KQzBnB7-U|Vocal Coaches Rea...|   38560|     225|  3645|      42430|
|SBmeEQOh20A|The Best FREE Sof...|  170837|     318|  5556|     176711|
|SBmeEQOh20A|The Best FREE Sof...|  170837|     318|  5556|     176711|
|eIrMbAQSU34|Java Tutorial for...| 7569833|    7321|195700|    7772854|
|eIrMbAQSU34|Java Tutorial for...| 7569833|    7321|195700|    7772854|
|casK0pXZ5sc|How to Simplify Y...|  153395|     314|  8227|     161936|
|casK0pXZ5sc|How to Simplify Y...|  153395|     314|  8227|     161936|
+-----------+--------------------+--------+--------+------+-----

In [44]:
join_video_comments_otimizado.write.mode('overwrite').parquet('join-video_comments-otimizado')